In [1]:
#!/usr/bin/env python3

# Phase 2 — Step 2: Tokenize Nepali Corpus → Binary Files

In [2]:
LANG = "nepali"
INPUT_DIR = "/kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/splits"
TOKENIZER_PATH = "/kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/tokenizer/tokenizer.model"
OUTPUT_DIR = "/kaggle/working/nepali_tokenized"
BOS_ID = 2
EOS_ID = 3
IS_JSONL = False

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])

0

In [4]:
import os, json, time, logging
import numpy as np
import sentencepiece as spm

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s")
logger = logging.getLogger("tokenize")


def tokenize_split(sp, input_path, output_path, bos_id=2, eos_id=3, is_jsonl=False):
    all_ids = []
    num_docs = 0
    logger.info(f"Tokenizing {input_path} ...")
    start = time.time()
    
    with open(input_path, "r", encoding="utf-8") as f:
        if is_jsonl:
            for line in f:
                line = line.strip()
                if not line: continue
                try: text = json.loads(line).get("text", "").strip()
                except json.JSONDecodeError: continue
                if not text: continue
                ids = sp.encode(text, out_type=int)
                all_ids.append(bos_id); all_ids.extend(ids); all_ids.append(eos_id)
                num_docs += 1
                if num_docs % 500_000 == 0:
                    logger.info(f"  {num_docs:,} docs, {len(all_ids):,} tokens")
        else:
            current_doc = []
            for line in f:
                stripped = line.strip()
                if stripped:
                    current_doc.append(stripped)
                else:
                    if current_doc:
                        text = " ".join(current_doc)
                        ids = sp.encode(text, out_type=int)
                        all_ids.append(bos_id); all_ids.extend(ids); all_ids.append(eos_id)
                        num_docs += 1; current_doc = []
                        if num_docs % 500_000 == 0:
                            logger.info(f"  {num_docs:,} docs, {len(all_ids):,} tokens")
            if current_doc:
                text = " ".join(current_doc)
                ids = sp.encode(text, out_type=int)
                all_ids.append(bos_id); all_ids.extend(ids); all_ids.append(eos_id)
                num_docs += 1
    
    arr = np.array(all_ids, dtype=np.uint16)
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    arr.tofile(output_path)
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    logger.info(f"  Done: {num_docs:,} docs, {len(all_ids):,} tokens, {size_mb:.1f} MB, {time.time()-start:.1f}s")
    return {"num_docs": num_docs, "num_tokens": len(all_ids), "file_size_mb": round(size_mb, 2)}

In [5]:
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
logger.info(f"Loaded tokenizer: vocab_size={sp.get_piece_size()}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
stats = {}
for split in ["train", "val", "test"]:
    ext = ".jsonl" if IS_JSONL else ".txt"
    input_path = os.path.join(INPUT_DIR, f"{split}{ext}")
    if not os.path.exists(input_path):
        logger.warning(f"Not found: {input_path}, skipping"); continue
    stats[split] = tokenize_split(sp, input_path, os.path.join(OUTPUT_DIR, f"{split}.bin"), BOS_ID, EOS_ID, IS_JSONL)

with open(os.path.join(OUTPUT_DIR, "tokenization_stats.json"), "w") as f:
    json.dump(stats, f, indent=2)

[2026-08-28 05:05:21,688] INFO — Loaded tokenizer: vocab_size=4000
[2026-08-28 05:05:21,695] INFO — Tokenizing /kaggle/input/notebooks/srinivasu2005/nepali-tokenizer/splits/train.txt ...
[2026-08-28 05:06:03,411] INFO —   500,000 docs, 37,305,927 tokens
[2026-08-28 05:06:45,200] INFO —   1,000,000 docs, 74,720,772 tokens
[2026-08-28 05:07:27,122] INFO —   1,500,000 docs, 112,003,553 tokens
[2026-08-28 05:08:09,370] INFO —   2,000,000 docs, 149,380,570 tokens
[2026-08-28 05:08:51,559] INFO —   2,500,000 docs, 186,901,398 tokens
[2026-08-28 05:09:33,288] INFO —   3,000,000 docs, 224,407,133 tokens
[2026-08-28 05:10:15,201] INFO —   3,500,000 docs, 261,570,911 tokens
[2026-08-28 05:10:57,314] INFO —   4,000,000 docs, 299,103,645 tokens
[2026-08-28 05:11:39,766] INFO —   4,500,000 docs, 336,786,243 tokens
[2026-08-28 05:12:21,501] INFO —   5,000,000 docs, 374,281,284 tokens
[2026-08-28 05:13:03,132] INFO —   5,500,000 docs, 411,640,259 tokens
[2026-08-28 05:13:45,330] INFO —   6,000,000 do

In [6]:
print(f"\nNepali Tokenization Complete")
for split, s in stats.items():
    print(f"  {split}: {s['num_docs']:,} docs, {s['num_tokens']:,} tokens, {s['file_size_mb']} MB")
print(f"Output: {OUTPUT_DIR}/")
print("Next: Upload output as dataset, then run 04_train_nepali.py")


Nepali Tokenization Complete
  train: 11,593,318 docs, 868,298,607 tokens, 1656.15 MB
  val: 241,858 docs, 18,068,301 tokens, 34.46 MB
  test: 241,085 docs, 18,012,884 tokens, 34.36 MB
Output: /kaggle/working/nepali_tokenized/
Next: Upload output as dataset, then run 04_train_nepali.py
